In [1]:
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
%pip install --user --force-reinstall -q JPype1==1.5.2

import os
os.makedirs("lucene_jars", exist_ok=True)
%cd lucene_jars

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?
Note: you may need to restart the kernel to use updated packages.
/home/cs172/BlueskyCrawler/lucene_jars


/home/cs172/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
LUCENE = "8.11.3"
MAVEN = "https://repo1.maven.org/maven2/org/apache/lucene"

!wget -q -nc https://repo1.maven.org/maven2/org/ow2/asm/asm/7.2/asm-7.2.jar
!wget -q -nc https://repo1.maven.org/maven2/org/ow2/asm/asm-commons/7.2/asm-commons-7.2.jar

for mod in ["lucene-core", "lucene-analyzers-common", "lucene-queryparser", "lucene-queries", "lucene-expressions", "antlr4-runtime-4.5.1-1.jar"]:
    !wget -q -nc {MAVEN}/{mod}/{LUCENE}/{mod}-{LUCENE}.jar

!wget -q -nc https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.5.1-1/antlr4-runtime-4.5.1-1.jar

!ls -lh *.jar
%cd /home/cs172/BlueskyCrawler

-rw-rw-r-- 1 cs172 cs172 295K Aug 20  2015 antlr4-runtime-4.5.1-1.jar
-rw-rw-r-- 1 cs172 cs172 113K Sep 29  2019 asm-7.2.jar
-rw-rw-r-- 1 cs172 cs172  69K Sep 29  2019 asm-commons-7.2.jar
-rw-rw-r-- 1 cs172 cs172 1.8M Feb  8  2024 lucene-analyzers-common-8.11.3.jar
-rw-rw-r-- 1 cs172 cs172 3.5M Feb  8  2024 lucene-core-8.11.3.jar
-rw-rw-r-- 1 cs172 cs172  76K Feb  8  2024 lucene-expressions-8.11.3.jar
-rw-rw-r-- 1 cs172 cs172 373K Feb  8  2024 lucene-queries-8.11.3.jar
-rw-rw-r-- 1 cs172 cs172 374K Feb  8  2024 lucene-queryparser-8.11.3.jar
/home/cs172/BlueskyCrawler


In [3]:
import glob
print(glob.glob("lucene_jars/*"))

['lucene_jars/lucene-expressions-8.11.3.jar', 'lucene_jars/lucene-queryparser-8.11.3.jar', 'lucene_jars/lucene-queries-8.11.3.jar', 'lucene_jars/lucene-analyzers-common-8.11.3.jar', 'lucene_jars/antlr4-runtime-4.5.1-1.jar', 'lucene_jars/asm-commons-7.2.jar', 'lucene_jars/lucene-core-8.11.3.jar', 'lucene_jars/asm-7.2.jar']


In [4]:
import jpype, jpype.imports, glob

if not jpype.isJVMStarted():
    jars = glob.glob("lucene_jars/*.jar")
    jpype.startJVM(classpath=jars, convertStrings=True)

print("✅ JVM running · classpath loaded with", len(jars), "jars")

✅ JVM running · classpath loaded with 8 jars


---
## Inverted Index

### Implementation:

In [5]:
import shutil
from java.nio.file import Paths
from org.apache.lucene.analysis.standard import StandardAnalyzer
from org.apache.lucene.document import Document, Field, FieldType, StoredField, IntPoint, StringField, NumericDocValuesField
from org.apache.lucene.index import IndexWriter, IndexWriterConfig, IndexOptions, DirectoryReader, Term
from org.apache.lucene.store import NIOFSDirectory

In [6]:
print(os.getcwd())

/home/cs172/BlueskyCrawler


In [7]:
import hashlib

# hash function to create a unique post_id for each post
def make_post_id(text, timestamp):
    clean_text = str(text).strip()
    clean_text = " ".join(clean_text.split())

    clean_timestamp = str(timestamp).strip()
    
    post_str = f'{clean_text}||{clean_timestamp}'
    post_id = hashlib.md5(post_str.encode('utf-8')).hexdigest()

    return post_id

In [8]:
import json
from datetime import datetime
post_files = ["bluesky_clean_data/clean_conspiracy.json", "bluesky_clean_data/clean_paranormal.json", "bluesky_clean_data/clean_strange_earth.json", "bluesky_clean_data/clean_ufo.json"]
def create_index(index_dir, post_files):
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)
    os.makedirs(index_dir)

    store = NIOFSDirectory(Paths.get(index_dir))
    analyzer = StandardAnalyzer()
    config = IndexWriterConfig(analyzer)
    config.setOpenMode(IndexWriterConfig.OpenMode.CREATE)
    writer = IndexWriter(store, config)

    meta_type = FieldType()
    meta_type.setStored(True)
    meta_type.setTokenized(False)

    text_type = FieldType()
    text_type.setStored(True)
    text_type.setTokenized(True)
    text_type.setIndexOptions(IndexOptions.DOCS_AND_FREQS_AND_POSITIONS)
    
    total_num_posts = 0
    for file in post_files:
        with open(file, "r", encoding="utf-8") as f:
            file_name = file.split("/")[-1]
            cat_with_format = file_name.split("_")[-1]
            category = cat_with_format.split(".")[0]
            for post in f:
                if not post.strip():
                    continue
                
                data_line = json.loads(post)

                post_title = data_line.get('title', "")
                username = data_line.get("user", "")
                display_name = data_line.get("display_name", "") or ""
                post_url = data_line.get("post_url", "")
                post_text = data_line.get("text", "")
                num_replies = data_line.get("reply_count", "")
                top_5_replies = data_line.get("top_5_replies", [])
                num_likes = data_line.get("like_count", "")
                num_reposts = data_line.get("repost_count", "")
                image_titles = data_line.get("image_titles", [])

                raw_time = data_line.get("time", "")
                raw_time = str(raw_time).strip() if raw_time else ""
                if raw_time == "":
                    time_secs = 0
                else:
                    try:
                        date_time = datetime.strptime(raw_time, "%m/%d/%Y %H:%M")
                        time_secs = int(date_time.timestamp())
                    except:
                        print(f"date string in wrong format: {raw_time}, setting to 0.")
                        time_secs = 0

                doc = Document()
                doc.add(Field("title",              post_title,        meta_type))
                doc.add(Field("username",           username,          meta_type))
                doc.add(Field("display_name",       display_name,      meta_type))
                doc.add(Field("url",                post_url,          meta_type))
                doc.add(Field("raw_time",           raw_time,          meta_type))
                doc.add(Field("category",           category,          meta_type))

                if isinstance(top_5_replies, list) and len(top_5_replies) > 0:
                    for reply in top_5_replies:
                        # prevents indexing replies that have no text 
                        reply = str(reply).strip()
                        if reply:
                            doc.add(Field("top_5_replies", reply, text_type))

                if isinstance(image_titles, list):
                    for img in image_titles:
                        doc.add(Field("image_titles", str(img), meta_type))
                else:
                    doc.add(Field("image_titles", str(image_titles), meta_type))

                doc.add(Field("post_text",          post_text,         text_type))

                # storing as a StoredField ensures we can also filter results by numerical value of these fields
                doc.add(StoredField("num_replies",  int(num_replies)))
                doc.add(StoredField("num_likes",    int(num_likes)))
                doc.add(StoredField("num_reposts",  int(num_reposts)))

                doc.add(IntPoint("num_replies",     int(num_replies)))
                doc.add(IntPoint("num_likes",       int(num_likes)))
                doc.add(IntPoint("num_reposts",     int(num_reposts)))

                doc.add(NumericDocValuesField("num_replies", int(num_replies)))
                doc.add(NumericDocValuesField("num_likes",   int(num_likes)))
                doc.add(NumericDocValuesField("num_reposts", int(num_reposts)))

                doc.add(NumericDocValuesField("time_secs", time_secs))

                post_id = make_post_id(post_text, raw_time)
                doc.add(StringField("post_id", post_id, Field.Store.YES))
                writer.updateDocument(Term("post_id", post_id), doc)
                total_num_posts += 1


    writer.commit()
    writer.close()

    print(f"✅ Indexed {total_num_posts} documents to {index_dir}")

### Construction:

In [9]:
INDEX_DIR = "bluesky_index"
create_index(INDEX_DIR, post_files)

✅ Indexed 41824 documents to bluesky_index


---
## Search Index

In [10]:
from org.apache.lucene.search import IndexSearcher
from org.apache.lucene.queryparser.classic import QueryParser

def search(index_dir, query_str, field="post_text", top_k=5):
    store = NIOFSDirectory(Paths.get(index_dir))
    reader = DirectoryReader.open(store)
    searcher = IndexSearcher(reader)

    parser = QueryParser(field, StandardAnalyzer())
    query = parser.parse(query_str)

    hits = searcher.search(query, top_k).scoreDocs

    results = []
    for hit in hits:
        doc = searcher.doc(hit.doc)
        results.append({
            "score":            round(hit.score, 4),
            "category":         doc.get("category"),
            "username":         doc.get("username"),
            "display_name":     doc.get("display_name"),
            "title":            doc.get("title"),
            "url":              doc.get("url"),
            "time":             doc.get("time"),
            "likes":            doc.get("num_likes"),
            "reposts":          doc.get("num_reposts")
        })
    reader.close()
    return results


def show(results, query):
    print(f"\n🔎  Query: {query!r}    ({len(results)} hits)")
    print("─" * 78)
    for i, r in enumerate(results, 1):
        print(f"{i}. [{r['score']:.3f}]  r/{r['category']}  ·  {r['display_name']}  ·  {r['time']}")
        print(f"Username: {r['username']}")
        print(f"Title: {r['title']}")
        print(f"{r['url']}")
        print(f"Likes: {r['likes']}  ·  Reposts: {r['reposts']}")

In [11]:
for q in ["deep state", "ghost", "ghost sighting"]:
    show(search(INDEX_DIR, q), q)


🔎  Query: 'deep state'    (5 hits)
──────────────────────────────────────────────────────────────────────────────
1. [5.824]  r/conspiracy  ·  GlacierLil1776  ·  None
Username: glacierlily.bsky.social
Title: Epstein = Deep State...
https://bsky.app/profile/did:plc:5bm7bvck7kstbxp7pwzx6yki/post/3ml4ktnltvs27
Likes: 3  ·  Reposts: 2
2. [5.707]  r/conspiracy  ·  Padj  ·  None
Username: makesalotofsense.com
Title: The true Deep State...
https://bsky.app/profile/did:plc:3ukhhpu3ieorbyr446h6r3h3/post/3ml4yaconr22o
Likes: 2  ·  Reposts: 0
3. [5.707]  r/conspiracy  ·  Der Urberliner  ·  None
Username: derurberliner.bsky.social
Title: Irgendwas mit Deep S...
https://bsky.app/profile/did:plc:lnexaxlliaaf4imc6zjubkrl/post/3mkz2p5h7p22p
Likes: 1  ·  Reposts: 0
4. [5.707]  r/conspiracy  ·  Jan Krattiger  ·  None
Username: jankrattiger.bsky.social
Title: Deep State aber ande...
https://bsky.app/profile/did:plc:2nq5ed6aninvd5uxlssjcwgi/post/3mkugi4522s2z
Likes: 1  ·  Reposts: 0
5. [5.668]  r/conspir